# Панин Иван Михайлович
# Лабораторная 2

## 1. Сгенерировать симметричную квадратную вещественную матрицу и столбец свободных членов системы линейных алгебраических уравнений Ax=b размерности n = 6.

In [ ]:
import numpy as np
import pandas as pd

n = 6
rng = np.random.default_rng(67)

def generate_matrix(n: int, rng: np.random.Generator):
    M = rng.uniform(-3, 3, size=(n, n))
    A = (M + M.T) / 2
    for i in range(n):
        row_sum_without_diag = np.sum(np.abs(A[i])) - abs(A[i, i])
        A[i, i] = row_sum_without_diag + rng.uniform(2, 5)
    return A

A = generate_matrix(n, rng)
b = rng.uniform(-5, 5, size=n)

print("Матрица A:")
print(A)

print("\nСтолбец свободных членов b:")
print(b)

print("\nПроверка симметричности A:")
print(np.allclose(A, A.T))

print("\nПроверка строгого диагонального преобладания по строкам(для работы итерационных методов):")
diag_dominance = []
for i in range(n):
    diag = abs(A[i, i])
    off_diag_sum = np.sum(np.abs(A[i])) - diag
    diag_dominance.append(diag > off_diag_sum)
    print(f"Строка {i + 1}: |a_ii| = {diag:.6f}, сумма вне диагонали = {off_diag_sum:.6f}, условие выполнено: {diag > off_diag_sum}")

print("\nВсе строки удовлетворяют условию:", all(diag_dominance))

Матрица A:
[[ 9.58598252 -1.69430229  0.99370503 -1.4247081  -0.69007887 -0.7227306 ]
 [-1.69430229  7.67858445 -0.17420436  0.74505963 -0.66743719 -0.38409939]
 [ 0.99370503 -0.17420436  8.42361724  1.28237596  0.42189673 -1.36648639]
 [-1.4247081   0.74505963  1.28237596  7.89738522 -1.77483769 -0.51462593]
 [-0.69007887 -0.66743719  0.42189673 -1.77483769  6.68526925 -0.14509286]
 [-0.7227306  -0.38409939 -1.36648639 -0.51462593 -0.14509286  6.08073697]]

Столбец свободных членов b:
[ 1.30639871  4.53590107 -0.98605431 -1.83847959 -3.49068603  2.79583216]

Проверка симметричности A:
True

Проверка строгого диагонального преобладания по строкам(для работы итерационных методов):
Строка 1: |a_ii| = 9.585983, сумма вне диагонали = 5.525525, условие выполнено: True
Строка 2: |a_ii| = 7.678584, сумма вне диагонали = 3.665103, условие выполнено: True
Строка 3: |a_ii| = 8.423617, сумма вне диагонали = 4.238668, условие выполнено: True
Строка 4: |a_ii| = 7.897385, сумма вне диагонали = 5.741

## 2. Найти решение системы, применяя методы из лабораторной 1.
(т.к. Лаб1 была выполнена на языках C и ассемблера, для данного пункта будет использована библиотечная реализация метода Гаусса)

In [ ]:
x_exact = np.linalg.solve(A, b)

print("Эталонное решение x, полученное библиотечным методом:")
print(x_exact)

residual = A @ x_exact - b
print("\nНевязка A x - b:")
print(residual)
print("\nНорма невязки ||Ax - b||_inf =", np.linalg.norm(residual, ord=np.inf))

Эталонное решение x, полученное библиотечным методом:
[ 0.19482111  0.64745333  0.03196891 -0.34926274 -0.52152405  0.4890193 ]

Невязка A x - b:
[ 0.00000000e+00  0.00000000e+00  1.11022302e-16 -2.22044605e-16
 -4.44089210e-16  0.00000000e+00]

Норма невязки ||Ax - b||_inf = 4.440892098500626e-16


# 3. С точностью ε1 = 10−4 и ε2 = 10−12 по норме ∥·∥∞ найти решение системы, используя: а) метод Якоби; б) метод Зейделя.

In [ ]:
acc = [1e-4, 1e-12]

def jacobi_method(A, b, eps, x0=None, max_iter=100000):
    n = len(b)
    if x0 is None:
        x_old = np.zeros(n)
    else:
        x_old = x0.astype(float).copy()

    D = np.diag(np.diag(A))
    L = np.tril(A, k=-1)
    U = np.triu(A, k=1)

    B = -np.linalg.inv(D) @ (L + U)
    c = np.linalg.inv(D) @ b
    q = np.linalg.norm(B, ord=np.inf)

    history = []

    for k in range(1, max_iter + 1):
        x_new = B @ x_old + c
        diff_norm = np.linalg.norm(x_new - x_old, ord=np.inf)

        if q < 1:
            posterior_estimate = q / (1 - q) * diff_norm
        else:
            posterior_estimate = np.inf

        true_error = np.linalg.norm(x_new - x_exact, ord=np.inf)

        history.append({
            "k": k,
            "||x_k - x_(k-1)||_inf": diff_norm,
            "апостериорная оценка": posterior_estimate,
            "фактическая ошибка относительно эталона": true_error
        })

        if posterior_estimate <= eps:
            return x_new, k, posterior_estimate, true_error, pd.DataFrame(history)

        x_old = x_new

    raise RuntimeError("Метод Якоби не сошелся за max_iter итераций.")







def seidel_method(A, b, eps, x0=None, max_iter=100000):
    n = len(b)
    if x0 is None:
        x_old = np.zeros(n)
    else:
        x_old = x0.astype(float).copy()

    D = np.diag(np.diag(A))
    L = np.tril(A, k=-1)
    U = np.triu(A, k=1)

    B = -np.linalg.inv(D + L) @ U
    c = np.linalg.inv(D + L) @ b
    q = np.linalg.norm(B, ord=np.inf)

    history = []

    for k in range(1, max_iter + 1):
        x_new = x_old.copy()

        for i in range(n):
            sum_left = A[i, :i] @ x_new[:i]
            sum_right = A[i, i + 1:] @ x_old[i + 1:]
            x_new[i] = (b[i] - sum_left - sum_right) / A[i, i]

        diff_norm = np.linalg.norm(x_new - x_old, ord=np.inf)

        if q < 1:
            posterior_estimate = q / (1 - q) * diff_norm
        else:
            posterior_estimate = np.inf

        true_error = np.linalg.norm(x_new - x_exact, ord=np.inf)

        history.append({
            "k": k,
            "||x_k - x_(k-1)||_inf": diff_norm,
            "апостериорная оценка": posterior_estimate,
            "фактическая ошибка относительно эталона": true_error
        })

        if posterior_estimate <= eps:
            return x_new, k, posterior_estimate, true_error, pd.DataFrame(history)

        x_old = x_new

    raise RuntimeError("Метод Зейделя не сошелся за max_iter итераций.")


# Норма

D = np.diag(np.diag(A))
L = np.tril(A, k=-1)
U = np.triu(A, k=1)

B_jacobi = -np.linalg.inv(D) @ (L + U)
c_jacobi = np.linalg.inv(D) @ b

B_seidel = -np.linalg.inv(D + L) @ U
c_seidel = np.linalg.inv(D + L) @ b

q_jacobi = np.linalg.norm(B_jacobi, ord=np.inf)
q_seidel = np.linalg.norm(B_seidel, ord=np.inf)

iteration_matrices_info = pd.DataFrame({
    "Метод": ["Якоби", "Зейделя"],
    "q = ||B||_inf": [q_jacobi, q_seidel],
    "Достаточное условие q < 1": [q_jacobi < 1, q_seidel < 1]
})

iteration_matrices_info


,Метод,q = ||B||_inf,Достаточное условие q < 1
0,Якоби,0.727026,True
1,Зейделя,0.576417,True


# 4. Используя априорные оценки, предварительно вычислить количество необходимых итераций. Сравнить с фактическим количеством итераций, которое определяется апостериорными оценками.

In [ ]:
def apriori_iteration_count(q: float, x1_minus_x0_norm: float, eps: float):
#    q^k / (1 - q) * ||x1 - x0|| <= eps.
    if q >= 1:
        return np.inf

    if x1_minus_x0_norm == 0:
        return 0

    value = np.log(eps * (1 - q) / x1_minus_x0_norm) / np.log(q)
    return max(0, int(np.ceil(value)))




x0 = np.zeros(n)

x1_jacobi = B_jacobi @ x0 + c_jacobi
x1_seidel = B_seidel @ x0 + c_seidel

apriori_rows = []

for method_name, q, x1 in [
    ("Якоби", q_jacobi, x1_jacobi),
    ("Зейделя", q_seidel, x1_seidel)
]:
    x1_minus_x0_norm = inf_norm(x1 - x0)

    for eps in acc:
        apriori_k = apriori_iteration_count(q, x1_minus_x0_norm, eps)
        apriori_rows.append({
            "Метод": method_name,
            "eps": eps,
            "q = ||B||_inf": q,
            "||x1 - x0||_inf": x1_minus_x0_norm,
            "Априорное число итераций": apriori_k
        })

apriori_table = pd.DataFrame(apriori_rows)
apriori_table

,Метод,eps,q = ||B||_inf,||x1 - x0||_inf,Априорное число итераций
0,Якоби,1.000000e-04,0.727026,0.590721,32
1,Якоби,1.000000e-12,0.727026,0.590721,90
2,Зейделя,1.000000e-04,0.576417,0.620792,18
3,Зейделя,1.000000e-12,0.576417,0.620792,51


In [ ]:
iterative_results = []
histories = {}

for eps in acc:
    x_j, k_j, post_j, true_j, hist_j = jacobi_method(A, b, eps)
    x_s, k_s, post_s, true_s, hist_s = seidel_method(A, b, eps)

    histories[("Якоби", eps)] = hist_j
    histories[("Зейделя", eps)] = hist_s

    apriori_j = apriori_iteration_count(q_jacobi, np.linalg.norm(x1_jacobi - x0, ord=np.inf), eps)
    apriori_s = apriori_iteration_count(q_seidel, np.linalg.norm(x1_seidel - x0, ord=np.inf), eps)

    iterative_results.append({
        "Метод": "Якоби",
        "eps": eps,
        "Априорное число итераций": apriori_j,
        "Фактическое число итераций": k_j,
        "Апостериорная оценка": post_j,
        "Фактическая ошибка": true_j,
        "Решение": x_j
    })

    iterative_results.append({
        "Метод": "Зейделя",
        "eps": eps,
        "Априорное число итераций": apriori_s,
        "Фактическое число итераций": k_s,
        "Апостериорная оценка": post_s,
        "Фактическая ошибка": true_s,
        "Решение": x_s
    })

iterative_results_df = pd.DataFrame(iterative_results)
iterative_results_df_display = iterative_results_df.copy()
iterative_results_df_display["Решение"] = iterative_results_df_display["Решение"].apply(
    lambda x: np.array2string(x, precision=8, suppress_small=False)
)

iterative_results_df_display



,Метод,eps,Априорное число итераций,Фактическое число итераций,Апостериорная оценка,Фактическая ошибка,Решение
0,Якоби,1.000000e-04,32,11,4.742690e-05,1.238135e-05,[ 0.19482883 0.64745874 0.03196293 -0.349250...
1,Зейделя,1.000000e-04,18,7,2.229649e-05,3.915298e-06,[ 0.19482502 0.64745307 0.031966 -0.349260...
2,Якоби,1.000000e-12,90,33,5.208616e-13,1.568745e-13,[ 0.19482111 0.64745333 0.03196891 -0.349262...
3,Зейделя,1.000000e-12,51,18,3.168161e-13,5.587197e-14,[ 0.19482111 0.64745333 0.03196891 -0.349262...


In [ ]:
print("Эталонное решение:")
print(x_exact)

print("\nСравнение решений:")
for _, row in iterative_results_df.iterrows():
    print(f"\nМетод: {row['Метод']}, eps = {row['eps']}")
    print("Решение:", row["Решение"])
    print("||x_iter - x_exact||_inf =", np.linalg.norm(row["Решение"] - x_exact, ord=np.inf))

Эталонное решение:
[ 0.19482111  0.64745333  0.03196891 -0.34926274 -0.52152405  0.4890193 ]

Сравнение решений:

Метод: Якоби, eps = 0.0001
Решение: [ 0.19482883  0.64745874  0.03196293 -0.34925036 -0.52151492  0.4890214 ]
||x_iter - x_exact||_inf = 1.2381348475032361e-05

Метод: Зейделя, eps = 0.0001
Решение: [ 0.19482502  0.64745307  0.031966   -0.34926002 -0.52152276  0.48901936]
||x_iter - x_exact||_inf = 3.915297734191636e-06

Метод: Якоби, eps = 1e-12
Решение: [ 0.19482111  0.64745333  0.03196891 -0.34926274 -0.52152405  0.4890193 ]
||x_iter - x_exact||_inf = 1.5687451337953462e-13

Метод: Зейделя, eps = 1e-12
Решение: [ 0.19482111  0.64745333  0.03196891 -0.34926274 -0.52152405  0.4890193 ]
||x_iter - x_exact||_inf = 5.5871973714261e-14


# 5. Найти наибольшее по модулю собственное число и соответствующий ему собственный вектор матрицы A, используя степенной метод.

In [ ]:
def normalize_vector(x):
    norm = np.linalg.norm(x)
    if norm == 0:
        raise ValueError("Нельзя нормировать нулевой вектор.")
    return x / norm


def align_vector_sign(x_new, x_old):
    if np.dot(x_new, x_old) < 0:
        return -x_new
    return x_new


def power_method(A, eps=1e-12, max_iter=100000):
    n = A.shape[0]
    x_old = normalize_vector(np.ones(n))

    history = []

    for k in range(1, max_iter + 1):
        y = A @ x_old
        x_new = normalize_vector(y)
        x_new = align_vector_sign(x_new, x_old)

        lambda_new = (x_new @ A @ x_new) / (x_new @ x_new)
        diff = np.linalg.norm(x_new - x_old, ord=np.inf)

        history.append({
            "k": k,
            "lambda": lambda_new,
            "||x_k - x_(k-1)||_inf": diff
        })

        if diff <= eps:
            return lambda_new, x_new, k, pd.DataFrame(history)

        x_old = x_new

    raise RuntimeError("Степенной метод не сошелся за max_iter итераций.")


lambda_max_power, vector_max_power, iterations_power, history_power = power_method(A, eps=1e-12)

print("Степенной метод")
print("Наибольшее по модулю собственное число:")
print(lambda_max_power)
print("\nСоответствующий собственный вектор:")
print(vector_max_power)
print("\nЧисло итераций:")
print(iterations_power)

Степенной метод
Наибольшее по модулю собственное число:
11.456159524662846

Соответствующий собственный вектор:
[ 0.77787566 -0.44435734  0.16899835 -0.38713363  0.11109938 -0.08173159]

Число итераций:
236


# 6. Найти наименьшее по модулю собственное число и соответствующий собственный вектор, используя метод обратных итераций.

In [ ]:
def inverse_iteration_method(A, eps=1e-12, max_iter=100000):
    n = A.shape[0]
    x_old = normalize_vector(np.ones(n))

    history = []

    for k in range(1, max_iter + 1):
        y = np.linalg.solve(A, x_old)
        x_new = normalize_vector(y)
        x_new = align_vector_sign(x_new, x_old)

        lambda_new = (x_new @ A @ x_new) / (x_new @ x_new)
        diff = np.linalg.norm(x_new - x_old, ord=np.inf)

        history.append({
            "k": k,
            "lambda": lambda_new,
            "||x_k - x_(k-1)||_inf": diff
        })

        if diff <= eps:
            return lambda_new, x_new, k, pd.DataFrame(history)

        x_old = x_new

    raise RuntimeError("Метод обратных итераций не сошелся за max_iter итераций.")


lambda_min_inverse, vector_min_inverse, iterations_inverse, history_inverse = inverse_iteration_method(A, eps=1e-12)

print("Метод обратных итераций")
print("Наименьшее по модулю собственное число:")
print(lambda_min_inverse)
print("\nСоответствующий собственный вектор:")
print(vector_min_inverse)
print("\nЧисло итераций:")
print(iterations_inverse)

Метод обратных итераций
Наименьшее по модулю собственное число:
4.332971487201413

Соответствующий собственный вектор:
[ 0.37630436  0.20965218 -0.24425471  0.54478232  0.6385586   0.22413339]

Число итераций:
126


# 7. Найти собственные числа с помощью любой библиотеки линейной алгебры. Сравнить результаты.

In [ ]:
eigvals, eigvecs = np.linalg.eigh(A)

idx_abs_max = np.argmax(np.abs(eigvals))
idx_abs_min = np.argmin(np.abs(eigvals))

lambda_max_library = eigvals[idx_abs_max]
lambda_min_library = eigvals[idx_abs_min]

spectral_comparison = pd.DataFrame({
    "Величина": [
        "Наибольшее по модулю собственное число",
        "Наименьшее по модулю собственное число"
    ],
    "Итерационный метод": [
        lambda_max_power,
        lambda_min_inverse
    ],
    "Библиотечный метод": [
        lambda_max_library,
        lambda_min_library
    ],
    "Абсолютная разность": [
        abs(lambda_max_power - lambda_max_library),
        abs(lambda_min_inverse - lambda_min_library)
    ]
})

print("Все собственные числа матрицы A:")
print(eigvals)

spectral_comparison

Все собственные числа матрицы A:
[ 4.33297149  5.31595823  6.95430199  8.07716355 10.21502088 11.45615952]


,Величина,Итерационный метод,Библиотечный метод,Абсолютная разность
0,Наибольшее по модулю собственное число,11.456160,11.456160,1.776357e-15
1,Наименьшее по модулю собственное число,4.332971,4.332971,0.000000e+00


# 8. Вычислить число обусловленности, используя полученные значения собственных чисел. Вычислить число обусловленности, применяя методы из лабораторной 1. Сравнить результаты.

In [ ]:
cond_by_eigenvalues = np.max(np.abs(eigvals)) / np.min(np.abs(eigvals))
cond_2_library = np.linalg.cond(A, p=2)

A_inv = np.linalg.inv(A)
cond_inf_by_norms = inf_norm(A) * inf_norm(A_inv)
cond_inf_library = np.linalg.cond(A, p=np.inf)

condition_table = pd.DataFrame({
    "Способ вычисления": [
        "По собственным числам, cond_2",
        "Библиотечно, cond_2",
        "Через нормы ||A||_inf * ||A^-1||_inf",
        "Библиотечно, cond_inf"
    ],
    "Значение": [
        cond_by_eigenvalues,
        cond_2_library,
        cond_inf_by_norms,
        cond_inf_library
    ]
})


condition_table



,Способ вычисления,Значение
0,"По собственным числам, cond_2",2.643950
1,"Библиотечно, cond_2",2.643950
2,Через нормы ||A||_inf * ||A^-1||_inf,4.111355
3,"Библиотечно, cond_inf",4.111355


In [ ]:
print("diff1: ", abs(cond_by_eigenvalues-cond_2_library))
print("diff2: ", abs(cond_inf_by_norms-cond_inf_library))

diff1:  8.881784197001252e-16
diff2:  0.0
